## 03. Agents — `create_agent`

> **학습 목표**
> 1. **LangChain v1.0의 `create_agent`** 를 사용해 ReAct 에이전트를 한 줄로 구성한다.
> 2. **ReAct 루프**(Reason → Act → Observe)의 흐름을 시각화하고, 02번에서 직접 구현한 수동 루프와 비교한다.
> 3. `stream()`을 활용해 에이전트의 사고 과정을 실시간으로 관찰한다.
> 4. **`MemorySaver` 체크포인터**로 멀티턴 대화를 구현한다.
> 5. `response_format`(`ToolStrategy` / `ProviderStrategy`)을 사용한 구조화 출력을 적용한다.
> 6. `@dynamic_prompt` 미들웨어로 컨텍스트(사용자 권한 등)에 따른 동적 프롬프트 분기를 구현한다.
>
---

## LangChain v1.0의 변경사항

LangChain 0.x 시점에는 에이전트 생성 API가 분산되어 있어 선택에 혼란이 있었습니다.

- `AgentExecutor` + `create_react_agent` / `create_tool_calling_agent`
- `langgraph.prebuilt.create_react_agent`

v1.0부터는 `langchain.agents.create_agent` 단일 함수로 통합되었습니다. 내부적으로 LangGraph 위에서 동작하므로 **체크포인팅, 스트리밍, HITL(Human-in-the-Loop), 상태 시간여행(time travel)** 기능이 기본으로 제공됩니다.

> ** 수동 루프와의 비교**: 02번에서 직접 구현한 `run_tool_loop`는 약 30줄의 함수였습니다. 본 노트북에서는 `create_agent(model=llm, tools=tools, system_prompt="...")` 단일 호출로 동일한 기능에 더해 메모리, 스트리밍, 가드레일까지 함께 제공받습니다.

---

## 1. 환경 설정

이전 노트북과 동일한 LLM 초기화 패턴을 사용합니다. 보유한 키에 따라 자동으로 제공자가 분기됩니다.

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

TAVILY_OK = bool(os.getenv("TAVILY_API_KEY"))
print(f"TAVILY_API_KEY 설정: {'✓' if TAVILY_OK else '✗ (폴백 시뮬레이션 사용)'}")

print(">> API Key 감지:",
      "Gemini" if os.getenv("GOOGLE_API_KEY") else "—",
      "Groq" if os.getenv("GROQ_API_KEY") else "—",
      "OpenAI" if os.getenv("OPENAI_API_KEY") else "—",
      "Upstage" if os.getenv("UPSTAGE_API_KEY") else "—")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")

TAVILY_API_KEY 설정: ✓
>> API Key 감지: Gemini Groq OpenAI Upstage


In [18]:

from langchain.chat_models import init_chat_model

#  Option 1: Google Gemini 3.1 Flash-Lite (무료)
#llm = init_chat_model("google_genai:gemini-3.1-flash-lite", temperature=0.3)
# llm = init_chat_model(
#     "gemini-3.1-flash-lite",
#     model_provider="google_genai",
#     temperature=0.3,
# )

#  Option 2: Groq (무료, 매우 빠름) — langchain-groq 대신 OpenAI 호환 엔드포인트 사용
# llm = init_chat_model(
#     "openai/gpt-oss-120b",
#     model_provider="openai",
#     base_url="https://api.groq.com/openai/v1",
#     api_key=GROQ_API_KEY,
#     temperature=0.3,
# )

#  Option 3: Upstage
llm = init_chat_model(
    "solar-pro3",
    model_provider="upstage",
    api_key=UPSTAGE_API_KEY,
    temperature=0.3,
)

#  Option 4: Ollama Qwen2.5 (완전 무료, 로컬)
# llm = init_chat_model(
#     "qwen2.5:1.5b",
#     model_provider="ollama",
#     temperature=0.3,
# )

#  Option 5: OpenAI (유료)
#llm = init_chat_model(model_provider="openai", model="gpt-4.1-mini", temperature=0.3)

print(type(llm), llm.model_name)
print(" LLM 준비 완료")


c:\myclass\ai_langchain\mylangchin-uv-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<class 'langchain_upstage.chat_models.ChatUpstage'> solar-pro3
 LLM 준비 완료


## 2. 에이전트와 체인의 비교

| 구분 | 체인 (LCEL) | 에이전트 |
|---|---|---|
| 처리 흐름 | **고정** (예: prompt → llm → parser) | **동적** — 모델이 도구 호출 결과를 보고 다음 행동을 결정 |
| 제어 주체 | 개발자가 처리 순서를 코딩 | 모델이 상황에 따라 자율 결정 |
| 도구 사용 | 수동 호출 필요 | 자동 (ReAct 루프) |
| 적합한 사례 | 단순 변환 파이프라인 | 복합 질의, 외부 시스템 조회 필요 작업 |

### ReAct(Reason → Act → Observe) 패턴

```mermaid
flowchart TD
    IN(["💬 사용자 질문<br/>HumanMessage"]):::input
    R["🧠 REASON<br/>LLM이 필요한 도구를 판단"]:::reason
    A["⚙️ ACT<br/>호출 측 코드가 도구 실행<br/>tool.invoke#40;args#41;"]:::act
    O["👁️ OBSERVE<br/>LLM이 결과를 바탕으로<br/>다음 행동 결정"]:::observe
    CHK{"추가 도구<br/>호출 필요?"}:::decision
    OUT(["✅ 최종 답변<br/>AIMessage"]):::output

    IN  --> R
    R   -->|"tool_calls"| A
    A   -->|"ToolMessage"| O
    O   --> CHK
    CHK -->|"Yes"| R
    CHK -->|"No"| OUT

    classDef input    fill:#4f46e5,stroke:#3730a3,color:#fff
    classDef reason   fill:#0f172a,stroke:#6366f1,color:#a5b4fc
    classDef act      fill:#0f172a,stroke:#f59e0b,color:#fcd34d
    classDef observe  fill:#0f172a,stroke:#10b981,color:#6ee7b7
    classDef decision fill:#1e293b,stroke:#94a3b8,color:#e2e8f0
    classDef output   fill:#059669,stroke:#047857,color:#fff
```

02번 노트북에서 직접 구현한 `run_tool_loop`가 정확히 이 흐름을 따랐습니다. `create_agent`는 동일한 루프를 단일 호출로 대체하면서, 메모리·스트리밍·가드레일 기능을 함께 제공합니다.

### ReAct 4단계 요약

1. **Reason**: 모델이 사용자 질문을 분석하고 필요한 도구를 판단합니다.
2. **Act**: `tool_calls`로 도구 호출을 요청합니다.
3. **Observe**: 도구 실행 결과를 `ToolMessage`로 받아 다음 판단의 근거로 활용합니다.
4. 추가 작업이 필요하면 1~3을 반복하고, 그렇지 않으면 최종 답변을 생성합니다.

## 3. `create_agent` 시그니처

```python
create_agent(
    model,                    # str 또는 BaseChatModel (예: "gpt-4.1-mini" 또는 llm 인스턴스)
    tools=[...],              # @tool 함수, 일반 함수, 또는 dict (빌트인 프로바이더 도구)
    *,
    system_prompt=None,       # str 또는 None. v0.x의 prompt= 가 v1.0에서 system_prompt= 로 변경됨
    middleware=(),            # AgentMiddleware 리스트
    response_format=None,     # ToolStrategy(Schema) 또는 ProviderStrategy(Schema)
    state_schema=None,        # 사용자 정의 TypedDict 상태
    context_schema=None,      # 런타임 컨텍스트 dataclass
    checkpointer=None,        # MemorySaver 등 — 대화 이력 저장
    store=None,               # 장기 메모리 스토어
    interrupt_before=None,    # HITL
    interrupt_after=None,     # HITL
    debug=False,
    name=None,
    cache=None,
)
```

## 4. 기본 에이전트 구성

02번 노트북의 도구 일부를 재사용해, 단일 호출로 에이전트를 구성합니다.

In [19]:
from datetime import datetime
from langchain.tools import tool

@tool
def get_schedule(date: str) -> list[dict]:
    """주어진 날짜(YYYY-MM-DD)의 사내 일정을 조회합니다."""
    fake_db = {
        "2026-04-15": [
            {"time": "10:00", "title": "주간 스탠드업", "room": "A"},
            {"time": "14:00", "title": "고객 데모", "room": "Zoom"},
        ],
        "2026-04-16": [
            {"time": "09:30", "title": "OKR 리뷰", "room": "대회의실"},
            {"time": "18:30", "title": "고객과의 식사", "room": "미정"}
        ],
    }
    return fake_db.get(date, [])


def count_business_days(start: str, end: str) -> int:
    """두 날짜(YYYY-MM-DD, 양 끝 포함) 사이의 영업일(주말 제외) 수를 계산한다.
    도구(calc_leave_days)와 구조화 출력 스키마(LeaveReport)가 같은 로직을 쓰도록 분리했다."""
    d1 = datetime.strptime(start, "%Y-%m-%d").date()
    d2 = datetime.strptime(end, "%Y-%m-%d").date()
    return sum(1 for i in range((d2 - d1).days + 1)
               if (d1.fromordinal(d1.toordinal() + i)).weekday() < 5)


@tool
def calc_leave_days(start: str, end: str) -> dict:
    """두 날짜 사이의 영업일(주말 제외)을 계산합니다."""
    return {"start": start, "end": end, "business_days": count_business_days(start, end)}


@tool
def get_leave_balance(employee_id: str) -> dict:
    """사원 잔여 연차를 조회합니다."""
    db = {"E12345": 13.5, "E67890": 7.0}
    return {"employee_id": employee_id, "remaining_days": db.get(employee_id, 10.0)}


# 웹검색 (실제 또는 폴백)
if TAVILY_OK:
    from langchain_tavily import TavilySearch
    web_search = TavilySearch(max_results=3)
    print("✓ 실제 Tavily 웹검색 도구 활성화")
else:
    @tool
    def web_search(query: str) -> list[dict]:
        """웹 검색 (폴백 시뮬레이션)."""
        return [{"title": f"(mock) {query}", "url": "https://example.com",
                 "content": f"mock result for {query}"}]

tools = [get_schedule, calc_leave_days, get_leave_balance, web_search]
print(f"tools: {[t.name for t in tools]}")

✓ 실제 Tavily 웹검색 도구 활성화
tools: ['get_schedule', 'calc_leave_days', 'get_leave_balance', 'tavily_search']


In [5]:
from langchain.agents import create_agent

#  핵심: 한 줄로 에이전트 생성
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "당신은 회사의 사내 업무 어시스턴트입니다. "
        "일정/휴가/웹검색 도구를 적절히 활용해 사용자 질문에 정확히 답하세요. "
        "날짜 계산이 필요하면 calc_leave_days 도구를 사용하세요. "
        "장소 추천처럼 웹 검색이 필요하면 날짜가 아니라 지역과 목적을 담은 구체적인 키워드로 검색하세요. "
        "(예: '강남 저녁 식사 장소')"
    ),
)

print(type(agent).__name__)
print("에이전트 준비 완료")

CompiledStateGraph
에이전트 준비 완료


### 에이전트 호출

입력 형식은 메시지 리스트입니다(`{"messages": [...]}`). 출력에는 사용자 메시지부터 모델 응답, 도구 호출 메시지가 순서대로 누적됩니다.

In [22]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "2026년 4월 15일 내 일정을 알려줘."}]
})

# 모든 메시지 확인 (사용자 → 도구 호출 → 도구 결과 → 최종 답변)
for m in result["messages"]:
    kind = type(m).__name__
    if hasattr(m, "tool_calls") and m.tool_calls:
        print(f"1.[{kind}] tool_calls: {m.tool_calls}")
    elif kind == "ToolMessage":
        print(f"2.[{kind}] {m.content[:100]}")
    else:
        print(f"3.[{kind}] {m.content[:200]}")

3.[HumanMessage] 2026년 4월 15일 내 일정을 알려줘.
1.[AIMessage] tool_calls: [{'name': 'get_schedule', 'args': {'date': '2026-04-15'}, 'id': '0g02983m59', 'type': 'tool_call'}]
2.[ToolMessage] [{"time": "10:00", "title": "주간 스탠드업", "room": "A"}, {"time": "14:00", "title": "고객 데모", "room": "Zo
3.[AIMessage] 2026년 4월 15일(목)의 사내 일정은 다음과 같습니다.

| 시간 | 일정 내용 | 장소 |
|------|-----------|------|
| 10:00 | 주간 스탠드업 | A 회의실 |
| 14:00 | 고객 데모 | Zoom 온라인 |

추가로 해당 날짜에 연차가 남아 있는지 확인해 드릴까요? (예: 연차 잔액 조회)


최종 답변만 추출하려면 마지막 메시지의 `content`를 사용합니다.

In [23]:
for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

2026년 4월 15일 내 일정을 알려줘.
================================== Ai Message ==================================
Tool Calls:
  get_schedule (0g02983m59)
 Call ID: 0g02983m59
  Args:
    date: 2026-04-15
================================= Tool Message =================================
Name: get_schedule

[{"time": "10:00", "title": "주간 스탠드업", "room": "A"}, {"time": "14:00", "title": "고객 데모", "room": "Zoom"}]
================================== Ai Message ==================================

2026년 4월 15일(목)의 사내 일정은 다음과 같습니다.

| 시간 | 일정 내용 | 장소 |
|------|-----------|------|
| 10:00 | 주간 스탠드업 | A 회의실 |
| 14:00 | 고객 데모 | Zoom 온라인 |

추가로 해당 날짜에 연차가 남아 있는지 확인해 드릴까요? (예: 연차 잔액 조회)


In [21]:
print(result["messages"][-1].content)

2026년 4월 15일(수) 일정은 다음과 같습니다:

| 시간 | 내용 | 장소 |
|------|------|------|
| 10:00 | 주간 스탠드업 | 회의실 A |
| 14:00 | 고객 데모 | Zoom |

필요한 사항이 있으면 알려 주세요!


## 5. 복합 질문 — 다중 도구의 자율 조합

에이전트의 강점은 **여러 도구를 자율적으로 조합**해야 답변 가능한 복합 질문에서 드러납니다. 모델이 처리 순서를 스스로 결정하고, 각 도구의 결과를 다음 단계의 입력으로 활용합니다.

In [25]:
query = (
    "사번 E12345 인 직원이 2026-04-20부터 2026-04-24까지 휴가를 쓰려고 해. "
    "사용 가능한 연차일이 충분한지 계산해서 알려줘."
)
result = agent.invoke({"messages": [{"role": "user", "content": query}]})
print(result["messages"][-1].content)

사번 **E12345**인 직원의 현재 잔여 연차는 **13.5일**입니다.  

- **휴가 예정 기간**: 2026‑04‑20 ~ 2026‑04‑24 (5일)  
- **필요 연차**: 5일  

잔여 연차가 13.5일이므로, 5일을 사용해도 충분히 남아 있습니다.  
남은 연차는 **8.5일**이 됩니다.  

추가로 필요하신 정보(예: 연차 사용 신청 절차, 휴가 승인 일정 등)가 있으면 알려 주세요!


In [26]:
for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

사번 E12345 인 직원이 2026-04-20부터 2026-04-24까지 휴가를 쓰려고 해. 사용 가능한 연차일이 충분한지 계산해서 알려줘.
================================== Ai Message ==================================
Tool Calls:
  get_leave_balance (979z5j4b2g)
 Call ID: 979z5j4b2g
  Args:
    employee_id: E12345
================================= Tool Message =================================
Name: get_leave_balance

{"employee_id": "E12345", "remaining_days": 13.5}
================================== Ai Message ==================================

사번 **E12345**인 직원의 현재 잔여 연차는 **13.5일**입니다.  

- **휴가 예정 기간**: 2026‑04‑20 ~ 2026‑04‑24 (5일)  
- **필요 연차**: 5일  

잔여 연차가 13.5일이므로, 5일을 사용해도 충분히 남아 있습니다.  
남은 연차는 **8.5일**이 됩니다.  

추가로 필요하신 정보(예: 연차 사용 신청 절차, 휴가 승인 일정 등)가 있으면 알려 주세요!


## 6. 스트리밍 — 처리 과정 실시간 관찰

`agent.stream(..., stream_mode="values")` 또는 `stream_mode="updates"`를 사용하면 에이전트의 처리 단계를 실시간으로 관찰할 수 있습니다. 디버깅과 사용자 인터페이스의 응답성 향상 모두에 활용됩니다.

In [27]:
print("=" * 60)
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "2026년 4월 16일 오후 일정 알려줘. 오후 일정에 강남 주변의 적합한 장소를 추천해 주세요"}]},
    stream_mode="values",
):
    last = chunk["messages"][-1]
    kind = type(last).__name__
    print(f"kind: {kind}")
    if hasattr(last, "tool_calls") and last.tool_calls:
        # 한 번에 여러 도구를 요청할 수 있으므로 모두 출력한다
        for tc in last.tool_calls:
            print(f" REASON → ACT: {tc['name']}({tc['args']})")
    elif kind == "ToolMessage":
        # 검색 결과는 길어서 앞부분만 출력 (results 내용까지 보이도록 충분히 길게)
        print(f" OBSERVE: {last.content[:500]}")
    elif kind == "AIMessage":
        # 최종 답변은 잘라내지 않고 전체를 출력한다
        print(" ANSWER:")
        print(last.content)
    elif kind == "HumanMessage":
        print(f" USER: {last.content}")
    print("-" * 60)

kind: HumanMessage
 USER: 2026년 4월 16일 오후 일정 알려줘. 오후 일정에 강남 주변의 적합한 장소를 추천해 주세요
------------------------------------------------------------
kind: AIMessage
 REASON → ACT: get_schedule({'date': '2026-04-16'})
------------------------------------------------------------
kind: ToolMessage
 OBSERVE: [{"time": "09:30", "title": "OKR 리뷰", "room": "대회의실"}, {"time": "18:30", "title": "고객과의 식사", "room": "미정"}]
------------------------------------------------------------
kind: AIMessage
 REASON → ACT: tavily_search({'query': '강남 저녁 식사 장소 추천', 'search_depth': 'advanced', 'include_images': False, 'topic': 'general'})
------------------------------------------------------------
kind: ToolMessage
 OBSERVE: {"query": "강남 저녁 식사 장소 추천", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.tripadvisor.co.kr/Restaurants-g294197-zfn7778634-zfp58-Seoul.html", "title": "강남/논현/역삼(서울) 소재 베스트 저녁식사 음식점 (10 곳)", "content": "강남/논현/역삼의 저녁식사 음식점 · 1. 양국. 4.7. 풍선 5개 중 4.7. (15

<!-- VIDEO: 3 / 메모리·체크포인트 -->

## 7. 메모리 — `MemorySaver` 체크포인터

`MemorySaver`는 `thread_id` 단위로 대화 이력을 자동 저장하는 체크포인터입니다.

- **`thread_id`**: 대화 세션을 식별하는 키. 동일 `thread_id`로 호출하면 이전 대화 이력이 자동 복원됩니다.
- **자동 저장**: 각 turn 종료 시 메시지 이력이 자동으로 체크포인트에 저장됩니다.
- **세션 격리**: 다른 `thread_id`로 호출하면 별개의 대화 세션으로 처리됩니다.

에이전트 생성 시 `checkpointer=MemorySaver()`를 지정하고, 호출 시 `config={"configurable": {"thread_id": "..."}}`를 전달하면 대화 이력이 `thread_id`별로 자동 관리됩니다.

In [28]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
agent_with_memory = create_agent(
    model=llm,
    tools=tools,
    system_prompt="당신은 사내 업무 어시스턴트입니다.",
    checkpointer=memory,
)

cfg = {"configurable": {"thread_id": "session-001"}}

# 첫 번째 턴
r1 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "내 사번은 E12345 이고, 이름은 길동이야."}]},
    config=cfg,
)
print("[1] ", r1["messages"][-1].content)
print()

# 두 번째 턴 — 이전 정보를 기억해야 함
r2 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "내 잔여 연차가 얼마나 남았지?"}]},
    config=cfg,
)
print("[2] ", r2["messages"][-1].content)

[1]  길동 님, 안녕하세요!  

사번 **E12345**의 현재 잔여 연차는 **13.5일**입니다.  

추가로 궁금하신 사항(예: 연차 사용 계획, 휴가 신청 방법 등)이 있으면 알려 주세요. 😊

[2]  길동 님, 현재 사번 **E12345**의 잔여 연차는 **13.5일** 남아 있습니다.  

추가로 필요하신 정보(예: 연차 사용 가능 기간, 휴가 신청 방법 등)가 있으면 알려 주세요!


### 다른 `thread_id`는 독립적인 대화 세션

새로운 `thread_id`로 호출하면 이전 대화 이력이 공유되지 않으며, 모델은 이전 맥락을 인지하지 못합니다.

In [29]:
cfg2 = {"configurable": {"thread_id": "session-002"}}
r3 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "내 사번 기억나?"}]},
    config=cfg2,
)
# 새로운 세션이므로 사번을 모를 것
print("[new session]", r3["messages"][-1].content)

[new session] 네, 기억합니다.  
제 사번은 **9527**입니다.  

혹시 사번을 확인하거나 다른 정보가 필요하신가요?


<!-- VIDEO: 4 / 구조화 출력 — ToolStrategy / ProviderStrategy -->

## 8. 구조화 출력 — `ToolStrategy` / `ProviderStrategy`

에이전트의 최종 답변을 **Pydantic 스키마에 맞춘 구조화 형식**으로 받고자 할 때 `response_format` 파라미터를 사용합니다. v1.0의 `create_agent`는 별도의 추가 LLM 호출 없이 기본 처리 루프 안에서 구조화 출력을 생성합니다.

| 전략 | 동작 방식 | 호환성 |
|---|---|---|
| `ToolStrategy(Schema)` | 스키마를 도구 형태로 모델에 노출하여, 모델이 해당 도구를 호출하는 방식으로 응답 | 도구 호출을 지원하는 모든 모델 |
| `ProviderStrategy(Schema)` | OpenAI, Anthropic 등 제공자의 **네이티브 구조화 출력 API** 활용 | 일부 최신 모델 (gpt-4.1, gpt-4o 계열 등) |

<br>

> **자동 폴백**: `response_format=MySchema`처럼 스키마만 전달하면, 모델이 네이티브 구조화 출력을 지원할 경우 `ProviderStrategy`로, 그렇지 않으면 `ToolStrategy`로 자동 선택됩니다.

결과는 `result["structured_response"]`에서 추출합니다.

### 8-1. `ToolStrategy` — 도구 호출과 구조화 출력의 결합

`get_leave_balance`, `calc_leave_days` 등의 도구를 자율적으로 호출하면서, 최종 답변은 `LeaveReport` 스키마 형식으로 반환받습니다.

같은 질문과 같은 도구를 쓰고 **스키마만 바꿔** 두 가지 방식을 비교합니다.

| 구분 | 스키마 | 값을 채우는 주체 |
|---|---|---|
| 8-1-a | `LeaveReportV1` | **LLM이 모든 필드를 직접 채움** (영업일 수 세기, 뺄셈, 승인 판단 포함) |
| 8-1-b | `LeaveReportV2` | LLM은 사번·날짜·요약만 채우고, **조회·계산은 도구(tool)가 처리** |

> **핵심**: LLM은 계산이나 조회를 틀리거나 건너뛸 수 있습니다. 정답이 정해진 값은 도구에 맡기면 LLM의 실수를 막을 수 있습니다.

#### 준비: 공통 설정과 점검 헬퍼

- 두 방식에서 **같은 질문과 같은 시스템 프롬프트**를 사용해, 결과 차이가 스키마에서만 생기도록 합니다.
- `check_report()`는 구조화 응답의 값과 **도구가 실제로 반환한 값**을 비교해 `✓ 일치` / `✗ 불일치`로 보여줍니다. 도구를 호출하지 않았다면 `도구 미호출`로 표시합니다.

In [20]:
import json
from datetime import datetime
from functools import cached_property

from pydantic import BaseModel, Field, computed_field, field_validator
from langchain.agents.structured_output import ToolStrategy, ProviderStrategy

# 두 방식(V1, V3)에서 공통으로 사용하는 질문과 시스템 프롬프트
LEAVE_QUESTION = "사번 E12345 가 2026-04-20부터 2026-04-24까지 휴가 쓰려고 합니다. 남은 승인 가능한지 알려주고 휴가 사용후에 남은 연차는 몇일인가요?"
LEAVE_SYSTEM_PROMPT = (
    "당신은 휴가 심사 보조원입니다. "
    "필요한 도구를 호출해 정보를 모은 뒤, 스키마에 맞춰 답하세요."
)


def check_report(result: dict) -> None:
    """구조화 응답의 값과 도구가 실제로 반환한 값을 비교해 일치 여부를 출력한다."""
    # ToolMessage에서 도구의 실제 반환값을 모은다 (구조화 응답용 ToolMessage는 제외)
    tool_vals = {}
    for m in result["messages"]:
        if m.type == "tool" and m.name in ("get_leave_balance", "calc_leave_days"):
            tool_vals[m.name] = json.loads(m.content)

    report = result["structured_response"]
    checks = {
        "remaining_days": (report.remaining_days,
                           tool_vals.get("get_leave_balance", {}).get("remaining_days")),
        "requested_business_days": (report.requested_business_days,
                                    tool_vals.get("calc_leave_days", {}).get("business_days")),
    }
    for name, (reported, actual) in checks.items():
        if actual is None:
            print(f"- 도구 미호출 | {name}: 응답={reported}")
        else:
            mark = "✓ 일치" if reported == actual else "✗ 불일치"
            print(f"{mark} | {name}: 응답={reported}, 도구={actual}")

#### 8-1-a. V1 — 모든 필드를 LLM이 채우면?

가장 단순한 방식입니다. 영업일 수와 휴가 후 남는 연차까지 **LLM이 직접 세고, 직접 뺄셈**합니다.

- 도구 결과를 보고 채우면 맞을 수 있지만, 아래와 같은 실수가 **실행마다** 생길 수 있습니다.
  - `calc_leave_days`를 호출하지 않고 영업일 수를 스스로 세서 채웁니다. 이번 실행에서도 `도구 미호출`로 확인됩니다. 같은 질문에서 5일로 나올 때도, 3일로 나올 때도 있었습니다.
  - 도구 결과를 받기 전에 스키마를 채워 잔여 연차를 추측합니다. (도구는 13.5를 반환했는데 15로 채운 사례가 있었습니다.)
- LLM 출력은 매번 달라 오류가 항상 재현되지는 않습니다. `check_report()`로 도구 반환값과 일치하는지 확인합니다.

In [21]:
class LeaveReportV1(BaseModel):
    """V1: 모든 필드를 LLM이 채운다."""
    employee_id: str = Field(description="사번")
    requested_business_days: int = Field(description="신청 영업일 수")
    remaining_days: float = Field(description="현재 잔여 연차")
    remaining_after_leave: float = Field(description="휴가 사용 후 남는 연차")
    can_approve: bool = Field(description="잔여 연차로 승인 가능한지 여부")
    summary: str = Field(description="한국어로 2~3문장 요약")


agent_v1 = create_agent(
    model=llm,
    tools=[get_leave_balance, calc_leave_days],
    system_prompt=LEAVE_SYSTEM_PROMPT,
    response_format=ToolStrategy(LeaveReportV1),
)

result_v1 = agent_v1.invoke({"messages": [{"role": "user", "content": LEAVE_QUESTION}]})
for msg in result_v1["messages"]:
    msg.pretty_print()

================================ Human Message =================================

사번 E12345 가 2026-04-20부터 2026-04-24까지 휴가 쓰려고 합니다. 남은 승인 가능한지 알려주고 휴가 사용후에 남은 연차는 몇일인가요?
================================== Ai Message ==================================
Tool Calls:
  get_leave_balance (z078gnd7ga)
 Call ID: z078gnd7ga
  Args:
    employee_id: E12345
  calc_leave_days (mq2ngl0xx9)
 Call ID: mq2ngl0xx9
  Args:
    end: 2026-04-24
    start: 2026-04-20
  LeaveReportV1 (cbw35h8foz)
 Call ID: cbw35h8foz
  Args:
    can_approve: True
    employee_id: E12345
    remaining_after_leave: 0
    remaining_days: 0
    requested_business_days: 5
    summary: E12345 사원은 2026-04-20부터 2026-04-24까지 5일간 휴가를 신청했으며, 현재 잔여 연차가 0일이므로 승인 불가능합니다.
================================= Tool Message =================================
Name: LeaveReportV1

Returning structured response: employee_id='E12345' requested_business_days=5 remaining_days=0.0 remaining_after_leave=0.0 can_approve=True summary='E12345 사원은 2026-04-2

In [22]:

print(result_v1["structured_response"].model_dump_json(indent=2))
print()
check_report(result_v1)  # 도구 반환값과 비교

{
  "employee_id": "E12345",
  "requested_business_days": 5,
  "remaining_days": 0.0,
  "remaining_after_leave": 0.0,
  "can_approve": true,
  "summary": "E12345 사원은 2026-04-20부터 2026-04-24까지 5일간 휴가를 신청했으며, 현재 잔여 연차가 0일이므로 승인 불가능합니다."
}

✗ 불일치 | remaining_days: 응답=0.0, 도구=13.5
✓ 일치 | requested_business_days: 응답=5, 도구=5


#### 8-1-b. V2 — 조회·계산은 도구로

LLM이 틀릴 수 있는 값은 LLM에게 채우게 하지 않고, **도구를 호출해 코드가 채웁니다.**

| 필드 | 채우는 주체 | 방법 |
|---|---|---|
| `employee_id`, `start_date`, `end_date`, `summary` | LLM | 질문 해석, 요약 문장 |
| `remaining_days` | 도구 | `get_leave_balance`로 사번 조회 |
| `requested_business_days` | 도구 | `calc_leave_days`로 영업일 계산 |
| `remaining_after_leave` | 코드 | `remaining_days - requested_business_days` |
| `can_approve` | 코드 | `remaining_days >= requested_business_days` |

- `@computed_field`는 **LLM에게 노출되지 않고**, 값이 채워진 뒤 코드가 계산합니다.
- `cached_property`로 도구는 필드당 한 번만 호출합니다.
- 날짜 형식이 틀리면 `field_validator`가 검증 오류를 내고, `ToolStrategy`가 LLM에게 다시 작성하도록 요청합니다.
- 따라서 `check_report()`는 `✓ 일치`가 됩니다.

> 학습용 예시라서 스키마 안에서 도구를 직접 호출합니다. 실제 서비스에서는 DB/사내 API를 호출하는 서비스 계층으로 바꿔야 합니다.
> `summary`는 여전히 LLM이 쓰는 문장이라, 문장 속 숫자가 계산값과 어긋날 가능성은 남아 있습니다.

In [25]:
class LeaveReportV2(BaseModel):
    """V2: 조회·계산은 도구(tool)가 처리하고, 해석(사번/날짜)과 요약만 LLM이 맡는다."""
    employee_id: str = Field(description="사번")
    start_date: str = Field(description="휴가 시작일 (YYYY-MM-DD)")
    end_date: str = Field(description="휴가 종료일 (YYYY-MM-DD)")
    summary: str = Field(description="한국어로 2~3문장 요약")

    @field_validator("start_date", "end_date")
    @classmethod
    def check_date_format(cls, v: str) -> str:
        """날짜 형식이 틀리면 검증 오류를 내서 LLM이 다시 작성하게 한다."""
        datetime.strptime(v, "%Y-%m-%d")
        return v

    @computed_field(description="현재 잔여 연차 (get_leave_balance 도구로 조회)")
    @cached_property
    def remaining_days(self) -> float:
        """사번으로 잔여 연차를 조회한다. (cached_property: 한 번만 조회)"""
        # TODO: 학습용으로 도구를 직접 호출한다. 실제 서비스에서는 DB/사내 API 조회 계층으로 교체
        return get_leave_balance.invoke({"employee_id": self.employee_id})["remaining_days"]

    @computed_field(description="신청 영업일 수 (주말 제외, calc_leave_days 도구로 계산)")
    @cached_property
    def requested_business_days(self) -> int:
        """시작일~종료일의 영업일 수를 도구로 계산한다. (LLM이 직접 세지 않는다)"""
        return calc_leave_days.invoke({"start": self.start_date, "end": self.end_date})["business_days"]

    @computed_field(description="휴가 사용 후 남는 연차")
    @property
    def remaining_after_leave(self) -> float:
        """잔여 연차 - 신청 영업일"""
        return self.remaining_days - self.requested_business_days

    @computed_field(description="잔여 연차로 승인 가능한지 여부")
    @property
    def can_approve(self) -> bool:
        """규칙: 잔여 연차 >= 신청 영업일이면 승인 가능"""
        return self.remaining_days >= self.requested_business_days


agent_v2 = create_agent(
    model=llm,
    tools=[get_leave_balance, calc_leave_days],
    system_prompt=LEAVE_SYSTEM_PROMPT + " summary에는 도구로 확인한 값만 사용하세요.",
    response_format=ToolStrategy(LeaveReportV2),
)

result_v2 = agent_v2.invoke({"messages": [{"role": "user", "content": LEAVE_QUESTION}]})

for msg in result_v2["messages"]:
    msg.pretty_print()

================================ Human Message =================================

사번 E12345 가 2026-04-20부터 2026-04-24까지 휴가 쓰려고 합니다. 남은 승인 가능한지 알려주고 휴가 사용후에 남은 연차는 몇일인가요?
================================== Ai Message ==================================
Tool Calls:
  get_leave_balance (z6v5eijgub)
 Call ID: z6v5eijgub
  Args:
    employee_id: E12345
  calc_leave_days (j61cmk5ekf)
 Call ID: j61cmk5ekf
  Args:
    end: 2026-04-24
    start: 2026-04-20
================================= Tool Message =================================
Name: get_leave_balance

{"employee_id": "E12345", "remaining_days": 13.5}
================================= Tool Message =================================
Name: calc_leave_days

{"start": "2026-04-20", "end": "2026-04-24", "business_days": 5}
================================== Ai Message ==================================
Tool Calls:
  get_leave_balance (8dhxhmzt0o)
 Call ID: 8dhxhmzt0o
  Args:
    employee_id: E12345
  calc_leave_days (hno1b6cl3j)
 Call ID: hno1

In [26]:
report: LeaveReportV2 = result_v2["structured_response"]
print("type:", type(report).__name__)
print(report.model_dump_json(indent=2))
print()
check_report(result_v2)  # 조회·계산을 도구가 하므로 항상 일치

type: LeaveReportV2
{
  "employee_id": "E12345",
  "start_date": "2026-04-20",
  "end_date": "2026-04-24",
  "summary": "사번 E12345가 2026-04-20부터 2026-04-24까지 5일간 휴가를 신청했습니다. 남은 연차는 13.5일이며, 휴가 사용 후 남은 연차는 8.5일입니다.",
  "remaining_days": 13.5,
  "requested_business_days": 5,
  "remaining_after_leave": 8.5,
  "can_approve": true
}

✓ 일치 | remaining_days: 응답=13.5, 도구=13.5
✓ 일치 | requested_business_days: 응답=5, 도구=5


#### 8-1 정리: 무엇을 도구로 옮길 것인가

| 점검 질문 | 예 | 처리 |
|---|---|---|
| LLM이 알 수 없는 값인가? (DB 값) | 잔여 연차 | 도구가 조회 |
| 규칙으로 정답이 하나인가? | 영업일 수, 뺄셈 | 도구/코드가 계산 |
| 규칙이 명확한 판단인가? | 승인 가능 여부 | 코드가 판단 |
| 해석·요약이 필요한가? | 사번/날짜 추출, 요약 문장 | LLM |

- 모든 값을 도구로 옮길 필요는 없습니다. **정확성이 중요한 값**만 옮기면 스키마와 도구의 결합도를 낮게 유지할 수 있습니다.
- 정리하면 **"조회·계산은 도구, 해석·요약은 LLM"** 입니다.

### 8-2. `ProviderStrategy` — 제공자 네이티브 구조화 출력

내부적으로 OpenAI의 `response_format={"type": "json_schema", ...}` 등 제공자 네이티브 API를 사용합니다. 모델이 해당 기능을 지원하는 경우 일반적으로 더 안정적이고 빠르게 동작하며, 도구 슬롯을 차지하지 않습니다.

아래 예시는 도구 없이 단순 정보 추출만 수행하는 경우입니다. 단, 다음 두 가지에 유의합니다.

- **모델 선택**: 네이티브 구조화 출력을 지원하는 모델이어야 합니다. 이 노트북의 기본 모델인 Upstage `solar-pro3`는 지원 여부가 확인되지 않아(`llm.profile`이 비어 있음), 이 예시에서는 OpenAI `gpt-4.1-mini`를 사용합니다.
- **빈 `tools=[]` 금지**: `ProviderStrategy`는 도구 목록을 그대로 `bind_tools`에 전달하므로, 비어 있으면 요청에 `tools: []`가 실립니다. API가 이를 거부(`400 Invalid 'tools': empty array`)하므로, 사용하지 않는 도구를 1개 전달하는 임시방편을 씁니다.

In [28]:
# ProviderStrategy는 제공자 네이티브 구조화 출력(response_format=json_schema)을 지원하는 모델이 필요하다.
# Upstage solar-pro3는 지원 여부가 확인되지 않아, 이 예시에서는 OpenAI 모델을 사용한다.
llm_openai = init_chat_model(
    "gpt-4.1-mini",
    model_provider="openai",
    api_key=OPENAI_API_KEY,
    temperature=0,
)


class ContactInfo(BaseModel):
    """고객 문의에서 추출한 연락처 정보."""
    name: str = Field(description="담당자 이름")
    email: str = Field(description="이메일 주소")
    phone: str = Field(description="전화번호 (하이픈 포함)")
    company: str | None = Field(default=None, description="회사명 (언급된 경우)")


agent_extract = create_agent(
    model=llm_openai,
    # TODO: 임시방편 — ProviderStrategy는 빈 tools 리스트를 요청에 그대로 실어 API가 400(empty array)으로 거부한다.
    #       실제로 호출되지 않는 도구를 1개 전달해 빈 배열을 피한다. (개선되면 tools=[]로 복원)
    tools=[get_schedule],
    system_prompt="고객 메시지에서 연락처를 정확히 추출하세요. 도구는 사용하지 마세요.",
    response_format=ProviderStrategy(ContactInfo),
)

msg = (
    "안녕하세요, AI전자회사의 교육팀 고길동 과장입니다. "
    "문의 답변은 gd.ko@gdko.co.kr 또는 010-1234-5678 로 부탁드립니다."
)

result = agent_extract.invoke({"messages": [{"role": "user", "content": msg}]})
contact: ContactInfo = result["structured_response"]
print(contact.model_dump_json(indent=2))

{
  "name": "고길동",
  "email": "gd.ko@gdko.co.kr",
  "phone": "010-1234-5678",
  "company": "AI전자회사"
}


### 8-3. 스키마만 전달하기 — 자동 전략 선택

v1.0에서는 `response_format=Schema`처럼 스키마만 전달해도 동작합니다. 모델이 네이티브 구조화 출력을 지원하면 `ProviderStrategy`, 그렇지 않으면 `ToolStrategy`로 **자동 선택**됩니다. 어떤 전략이 사용되는지 직접 관리할 필요가 없는 가장 간편한 방식입니다.

In [39]:
class ScheduleSummary(BaseModel):
    """하루 일정 요약."""
    date: str = Field(description="조회한 날짜 (YYYY-MM-DD)")
    total: int = Field(description="일정 개수")
    titles: list[str] = Field(description="일정 제목 리스트")


# 스키마만 전달 → 내부에서 적절한 전략 자동 선택
agent_auto = create_agent(
    model=llm,
    tools=[get_schedule],
    system_prompt="요청된 날짜의 일정을 조회해 ScheduleSummary 로 답하세요.",
    response_format=ScheduleSummary,
)

result = agent_auto.invoke({
    "messages": [{"role": "user", "content": "2026-04-15 일정 요약해줘"}]
})

summary: ScheduleSummary = result["structured_response"]
print(summary)
print("titles:", summary.titles)

date='2026-04-15' total=2 titles=['주간 스탠드업', '고객 데모']
titles: ['주간 스탠드업', '고객 데모']


### 8-4. 전략 선택 가이드

| 상황 | 권장 전략 |
|---|---|
| 프로토타이핑, 모델 교체 가능성을 고려해야 하는 경우 | `response_format=Schema` (자동 선택) |
| 안정성을 최우선으로 하며 OpenAI/Anthropic 최신 모델을 사용하는 경우 | `ProviderStrategy(Schema)` (도구 목록이 비어 있으면 400 에러가 날 수 있음, 8-2 참고) |
| 구형 또는 오픈소스 모델을 포함해 여러 제공자에서 동일하게 동작해야 하는 경우 | `ToolStrategy(Schema)` |
| 도구 호출과 구조화 출력을 동일 루프 내에서 처리해야 하는 경우 | 두 전략 모두 가능 (8-1은 `ToolStrategy` 사용) |

<br>

> 자동 선택(`response_format=Schema`)에서는 모델이 네이티브 구조화 출력을 지원하지 않거나, 일부 모델(Gemini 등)이 도구와 함께 쓸 수 없는 경우 `ToolStrategy`가 선택됩니다. `ProviderStrategy`를 명시하면 폴백 없이 그대로 사용되므로 모델 지원 여부를 직접 확인해야 합니다.
>
> 참고: 이 환경에서 자동 선택 결과는 Upstage `solar-pro3` → `ToolStrategy`, OpenAI `gpt-4.1-mini` → `ProviderStrategy`입니다.

<!-- VIDEO: 5 / @dynamic_prompt — 컨텍스트 기반 프롬프트 분기 -->

## 9. 동적 프롬프트 — `@dynamic_prompt` 미들웨어

런타임 컨텍스트(예: 사용자 권한, 언어 설정)에 따라 시스템 프롬프트를 다르게 적용해야 하는 경우, v1.0에서는 `middleware` 파라미터에 `@dynamic_prompt` 데코레이트된 함수를 등록하여 처리합니다.

> **v0.x와의 차이**: 0.x에서는 `prompt=callable` 형태로 함수를 직접 전달했으나, v1.0에서는 반드시 `@dynamic_prompt` 데코레이터로 감싼 후 `middleware=[...]`에 등록해야 합니다.

In [29]:
from dataclasses import dataclass
from langchain.agents.middleware import dynamic_prompt

#  ModelRequest 는 일부 버전에서 .types 서브모듈에 위치 — 두 경로 중 동작하는 것 사용
try:
    from langchain.agents.middleware import ModelRequest
except ImportError:
    from langchain.agents.middleware.types import ModelRequest


@dataclass
class UserContext:
    user_level: str = "user"       # 'user' 또는 'admin'
    user_name: str = ""
    employee_id: str = ""           # 도구 호출에 바로 사용


@dynamic_prompt
def personalized_prompt(request: ModelRequest) -> str:
    ctx: UserContext = request.runtime.context
    base = "당신은 사내 업무 어시스턴트입니다."

    if ctx.user_level == "admin":
        return (
            f"{base}\n"
            f"[호출자] {ctx.user_name} / 권한: **관리자** / 사번: {ctx.employee_id}\n"
            "[답변 규칙]\n"
            "- 반드시 '📊 [관리자 리포트]' 로 시작합니다.\n"
            "- 다음 항목을 **모두** 글머리표(•)로 포함합니다:\n"
            "   • 사번(raw 그대로)\n"
            "   • 잔여 연차(소수점 포함 정확한 숫자)\n"
            "   • 내부 정책 수치: 연차 1일 = 근무 8시간, 연간 부여 15일, 이월 한도 5일\n"
            "   • 타 직원 대비 상대 수준 (추정 코멘트 1줄)\n"
            "- 경어체는 유지하되 수치/PII 를 가리지 말고 투명하게 공개합니다.\n"
            "- 도구 호출 시 employee_id 는 호출자 사번을 그대로 사용하세요."
        )
    else:
        return (
            f"{base}\n"
            f"[호출자] {ctx.user_name} / 권한: 일반 직원 / 사번: {ctx.employee_id}\n"
            "[답변 규칙]\n"
            "- 반드시 ' [안내]' 로 시작하며 **한두 문장** 으로만 답합니다.\n"
            "- 사번·정확한 소수점·타 직원 정보·내부 정책 수치는 **절대 공개 금지**.\n"
            "- 잔여 연차는 '약 N일 정도' 처럼 **반올림한 자연어** 로만 표현합니다.\n"
            "- 팀 평균/타 직원 비교 요청은 정중히 거절하세요.\n"
            "- 도구 호출 시 employee_id 는 호출자 사번을 그대로 사용하세요."
        )


agent_dyn = create_agent(
    model=llm,
    tools=tools,
    middleware=[personalized_prompt],     # system_prompt 는 미들웨어가 제공
    context_schema=UserContext,
)

# 같은 질문 — "내 연차 + 팀원 평균 + 정책 상세" 를 한 번에 요청 → 권한별 응답이 확연히 갈림
question = (
    "제 잔여 연차를 알려주시고, 팀원 평균 대비 어느 정도인지, "
    "그리고 이월 한도 같은 내부 정책 수치도 함께 설명해 주세요."
)

# 일반 사용자 호출
r_user = agent_dyn.invoke(
    {"messages": [{"role": "user", "content": question}]},
    context=UserContext(user_level="user", user_name="고길동", employee_id="E67890"),
)
print("[USER]")
print(r_user["messages"][-1].content)
print()
print("-" * 60)
print()

# 관리자 호출
r_admin = agent_dyn.invoke(
    {"messages": [{"role": "user", "content": question}]},
    context=UserContext(user_level="admin", user_name="김부장", employee_id="E12345"),
)
print("[ADMIN]")
print(r_admin["messages"][-1].content)

[USER]
[안내] 현재 잔여 연차는 약 7일 정도 남아 있습니다. 팀원 평균 대비 연차 보유량이나 이월 한도와 같은 내부 정책 수치는 회사 규정에 따라 공개되지 않으며, 정확한 비교나 정책 정보는 인사팀을 통해 확인하시기 바랍니다.

------------------------------------------------------------

[ADMIN]
📊 [관리자 리포트]
• 사번: E12345
• 잔여 연차: 13.5일
• 내부 정책 수치: 연차 1일 = 근무 8시간, 연간 부여 15일, 이월 한도 5일
• 타 직원 대비 상대 수준: 현재 잔여 연차를 보유한 직원 중 상위 20% 이내에 해당하는 수준입니다.


---

## 실습 과제: 사내 업무 에이전트 구현

**요구사항**

1. 최소 3개의 `@tool`을 정의합니다 (예: 회의실 예약 조회, 구내식당 메뉴, 출장 승인 상태 등).
2. `create_agent`와 `MemorySaver`로 에이전트를 구성합니다.
3. 동일한 `thread_id`로 3회 이상 대화를 이어가며, 메모리가 정상 동작하는지 확인합니다.
4. 한 가지 질문에서는 반드시 **두 개 이상의 도구를 연쇄 호출**하도록 구성합니다.

In [16]:
# 여기에 코드를 작성하세요
# Hint:
# @tool
# def book_room(room: str, time: str) -> dict: ...
# @tool
# def get_lunch_menu(date: str) -> list[str]: ...
#
# my_agent = create_agent(model=llm, tools=[...], system_prompt="...", checkpointer=MemorySaver())
# cfg = {"configurable": {"thread_id": "my-session"}}
# my_agent.invoke({"messages": [...]}, config=cfg)


<details>
<summary>모범 답안 보기</summary>

```python
@tool
def book_room(room: str, time: str) -> dict:
    """회의실 예약을 시도합니다."""
    return {"status": "booked", "room": room, "time": time}

@tool
def get_lunch_menu(date: str) -> list[str]:
    """구내식당 점심 메뉴 조회 (하드코딩)."""
    return ["불고기 정식", "카레라이스", "비빔국수", "샐러드바"]

@tool
def check_trip_approval(trip_id: str) -> dict:
    """출장 승인 상태를 조회합니다."""
    db = {"T-001": "approved", "T-002": "pending"}
    return {"trip_id": trip_id, "status": db.get(trip_id, "unknown")}

my_tools = [book_room, get_lunch_menu, check_trip_approval]

my_agent = create_agent(
    model=llm,
    tools=my_tools,
    system_prompt="당신은 ABC 사내 업무 어시스턴트입니다. 필요한 도구를 자유롭게 조합해 사용자를 도우세요.",
    checkpointer=MemorySaver(),
)

cfg = {"configurable": {"thread_id": "exercise-001"}}

turns = [
    "내일 10시에 회의실 A 예약해줘",
    "그리고 그날 점심 메뉴도 알려줘",
    "마지막으로 출장 T-001 승인 상태 확인해줘",
]
for t in turns:
    r = my_agent.invoke({"messages": [{"role": "user", "content": t}]}, config=cfg)
    print(">", t)
    print("<", r["messages"][-1].content)
    print()
```

</details>

---

## 보너스 — 미니 여행 플래너 에이전트

다중 도구의 자율 조합을 실제 시나리오로 확인합니다. 항공권, 숙소, 환율 도구를 정의한 뒤, "도쿄 3박 4일" 같은 자연어 요청을 처리하도록 합니다.

In [30]:
@tool
def search_flight(origin: str, destination: str, date: str) -> list[dict]:
    """항공편을 검색합니다. 가격은 만원 단위입니다."""
    return [
        {"airline": "대한항공", "depart": "10:30", "arrive": "12:45", "price_man": 38},
        {"airline": "ANA", "depart": "13:15", "arrive": "15:25", "price_man": 32},
        {"airline": "Peach", "depart": "07:50", "arrive": "10:10", "price_man": 19},
    ]

@tool
def search_hotel(city: str, nights: int, max_price_per_night_man: int) -> list[dict]:
    """도시별 호텔을 검색합니다 (시뮬레이션)."""
    db = {
        "도쿄": [
            {"name": "신주쿠 비즈니스", "price_man": 12, "rating": 4.2},
            {"name": "시부야 모던", "price_man": 18, "rating": 4.5},
            {"name": "아사쿠사 료칸", "price_man": 25, "rating": 4.8},
        ],
        "오사카": [
            {"name": "난바 비즈니스", "price_man": 9, "rating": 4.0},
        ],
    }
    return [h for h in db.get(city, []) if h["price_man"] <= max_price_per_night_man]

@tool
def convert_jpy_to_krw(yen: int) -> dict:
    """엔(JPY) → 원(KRW) 환율 변환 (간단 시뮬레이션)."""
    return {"yen": yen, "krw": int(yen * 9.15)}

travel_agent = create_agent(
    model=llm,
    tools=[search_flight, search_hotel, convert_jpy_to_krw],
    system_prompt=(
        "당신은 여행 플래너입니다. 사용자의 예산·일정에 맞춰 항공/숙소를 추천하고, "
        "총 예상 경비를 만원 단위로 정리해 답하세요. 가성비와 평점을 함께 고려하세요."
    ),
)

q = "서울에서 도쿄로 2026년 5월 10일 출발해서 3박 4일 여행 갈 예정이야. 1인 80만원 예산으로 항공권 + 호텔 추천해줘."
result = travel_agent.invoke({"messages": [{"role": "user", "content": q}]})
print(result["messages"][-1].content)

**추천 일정 (서울 → 도쿄, 2026‑05‑10 출발, 3박 4일)**  

| 항목 | 선택 옵션 | 가격 (만원) | 평점 | 비고 |
|------|-----------|------------|------|------|
| 항공권 | **Peach** (저가 항공) | **19** | ★★★★☆ (4.0) | 07:50 출발 – 10:10 도착, 1시간 20분 비행 |
| 호텔 | **신주쿠 비즈니스** | **12** (1박) | ★★★★☆ (4.2) | 3박 → 총 **36**만원, 비즈니스급, 교통·식사 편리 |
| **총 예상 경비** |  | **55**만원 |  | 80만원 예산 안에서 여유 25만원 남음 |

### 왜 이 조합을 추천하나요?
1. **가성비**  
   - Peach 항공은 가장 저렴한 옵션(19만원)이며, 1시간 20분 비행으로 도쿄 도착이 빠릅니다.  
   - 신주쿠 비즈니스는 1박당 12만원으로, 3박이면 36만원. 같은 가격대에서 평점 4.2로 만족도가 높습니다.  

2. **편의성**  
   - **신주쿠**는 도쿄의 교통 허브이자 쇼핑·식당·문화 시설이 집중된 지역이라 이동과 관광이 편리합니다.  
   - 호텔은 비즈니스급이지만 깔끔한 객실과 무료 Wi‑Fi, 조식(옵션) 등을 제공해 여행 피로를 최소화합니다.  

3. **예산 여유**  
   - 전체 비용이 55만원으로, 80만원 예산 대비 25만원이 남아 식사·교통·여가 활동에 활용할 수 있습니다.  

### 추가 팁
- **교통**: 신주쿠에서 도쿄 메트로·JR 패스(24시간) 구매 시 1일당 약 8~10만원 정도 절감할 수 있습니다.  
- **식사**: 현지 편의점·라멘 가게, 혹은 1인당 1~2만원 정도의 레스토랑을 이용하면 예산을 크게 초과하지 않습니다.  
- **관광**: 신주쿠·고쿄·하라주쿠·아사쿠사 등 주요 명소를 하루씩 배분하면 3박 4일 동안 충분히 즐길 수 있습니다.  

필요하신 경우 다른 항공사·호텔 옵션(예: ANA 

---

## 트러블슈팅

| 증상 | 원인 | 해결 방법 |
|---|---|---|
| `ImportError: create_agent` | LangChain v0.x 사용 중 | `pip install -U langchain langgraph` (1.0 이상) |
| 에이전트가 무한 루프 | 모델이 동일 도구를 반복 호출 | `system_prompt`에 "이미 호출한 도구 결과를 그대로 사용"을 명시 |
| 메모리가 동작하지 않음 | `thread_id` 누락 | `config={"configurable": {"thread_id": "..."}}` 전달 필요 |
| `response_format` 미동작 | 모델이 네이티브 구조화 출력 미지원 | `ToolStrategy(Schema)` 명시적 사용 |
| Ollama 로컬 모델의 도구 호출 실패 | 일부 로컬 모델의 도구 호출 정확도 부족 | Gemini 또는 Groq 클라우드 모델로 전환 |
| `@dynamic_prompt`와 `system_prompt` 충돌 | 두 옵션 동시 사용 시 미들웨어가 우선 적용 | 둘 중 하나만 사용 |

---

## 마무리

본 노트북에서 학습한 내용은 다음과 같습니다.

- `create_agent` 단일 호출로 ReAct 에이전트 구성 (04번의 30줄 → 1줄로 축약)
- ReAct 루프 다이어그램 기반의 처리 흐름 이해 (Reason → Act → Observe)
- `agent.stream()`을 통한 사고 과정 실시간 관찰
- `MemorySaver`와 `thread_id`를 활용한 멀티턴 메모리 관리
- `response_format`(`ToolStrategy`, `ProviderStrategy`, 자동 선택) 기반의 구조화 출력
- `@dynamic_prompt` 미들웨어를 활용한 컨텍스트 기반 프롬프트 분기
- 미니 여행 플래너 에이전트 구현 사례

### 다음 노트북(06번) 예고

여기까지의 에이전트는 외부 도구(API)에 접근할 수 있었습니다. 그러나 사내 정책 PDF, 내부 매뉴얼과 같은 **비공개 문서**의 내용은 모델이 알지 못합니다.

다음 노트북에서는 **RAG(Retrieval-Augmented Generation)** 를 다룹니다. 문서를 분할하고 임베딩으로 의미 좌표계에 매핑한 뒤, 질의와 관련도가 높은 문서 조각만 모델에 컨텍스트로 제공하는 방법을 학습합니다.